# Ligand-Pocket QGNN: IBM Quantum Hardware vs Classical Comparison

This notebook compares **IBM Quantum Hardware** and **Classical** versions of the Ligand-Pocket QGNN model on the binding classification task.

**Architecture:**
- **Ligand**: Graph Neural Network (GCN) → Latent Vector
- **Pocket**: MLP → Latent Vector
- **Interaction**: Quantum Circuit on IBM Quantum Hardware or Classical MLP → Probability

**Task:** Binary Classification (Binding vs Non-Binding)

## Important Notes:
1. **IBM Quantum Account Required**: You need an IBM Quantum account and API token
2. **Setup Instructions**: Run the setup cell to configure your IBM Quantum credentials
3. **Backend Selection**: Choose from available IBM backends (simulators or real quantum computers)
4. **Reduced Batch Size**: IBM Quantum has API rate limits, so batch size is reduced
5. **Training Time**: Real quantum hardware will be significantly slower than simulators

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import multiprocessing as mp
from data import LigandPocketDataProcessor, LigandPocketDataset

# Use ORIGINAL model_ibm.py for IBM Quantum hardware access
from model_ibm import LigandPocketQGNN_IBM

# IBM Quantum imports
from qiskit_ibm_runtime import QiskitRuntimeService

# ========== OPTIMIZED COLLATE FUNCTION ==========
def optimized_collate_fn(batch):
    """
    Faster collate function using vectorized PyTorch operations.
    Replaces Python loops with batched tensor operations.
    """
    # Unzip batch
    x_list, edge_index_list, pocket_list, label_list = zip(*batch)

    # Fast concatenation
    pocket_batch = torch.stack(pocket_list)
    label_batch = torch.stack(label_list)

    # Calculate offsets vectorized
    num_nodes_list = torch.tensor([x.shape[0] for x in x_list], dtype=torch.long)
    cumsum = torch.cat([torch.zeros(1, dtype=torch.long), num_nodes_list.cumsum(0)])

    # Batch graphs
    x_batch = torch.cat(x_list, dim=0)

    # Shift edge indices vectorized
    edge_index_shifted = []
    for i, edge_index in enumerate(edge_index_list):
        if edge_index.shape[1] > 0:
            edge_index_shifted.append(edge_index + cumsum[i])

    if edge_index_shifted:
        edge_index_batch = torch.cat(edge_index_shifted, dim=1)
    else:
        edge_index_batch = torch.zeros((2, 0), dtype=torch.long)

    # Create batch vector (which sample each node belongs to)
    batch_vec = torch.cat([torch.full((n,), i, dtype=torch.long)
                           for i, n in enumerate(num_nodes_list)])

    return x_batch, edge_index_batch, batch_vec, pocket_batch, label_batch

print("✓ Optimized collate function loaded!")
print("✓ IBM Quantum model imported (ORIGINAL version for IBM hardware)")

In [ ]:
## IBM Quantum Setup

import os
from dotenv import load_dotenv
from qiskit_ibm_runtime import QiskitRuntimeService

# Try loading from .env file
env_loaded = load_dotenv()
print(f" .env file loaded: {env_loaded}")

# Try multiple possible environment variable names
Quantum_IBM = None
for key in ['Quantum_IBM', 'QUANTUM_IBM', 'IBM_QUANTUM_TOKEN', 'QISKIT_TOKEN']:
    val = os.getenv(key)
    if val:
        # Clean the token (remove quotes, whitespace)
        Quantum_IBM = val.strip().strip("'\"")
        print(f" Found IBM Quantum token in environment variable: {key}")
        print(f"  Token length: {len(Quantum_IBM)} characters")
        break

# If no token found, try to load existing saved account
if not Quantum_IBM:
    print(" No token found in environment variables")
    print("  Attempting to load existing saved account...")
    
    try:
        # Try to load existing saved account
        service = QiskitRuntimeService()
        print(" IBM Quantum credentials loaded from saved configuration!")
        print(f" Available backends: {len(service.backends())}")
        
        # List available backends
        print("\nAvailable IBM Quantum Backends:")
    
        for backend in service.backends():
            status = backend.status()
            print(f"  {backend.name:30s} - Qubits: {backend.num_qubits:3d} - Status: {status.status_msg}")
    
    except Exception as e:
        print(f" Error: {e}")
        raise EnvironmentError(
            "\nIBM Quantum token not found!\n"
            "Please set 'Quantum_IBM' in your .env file with your IBM Quantum token.\n"
            "Get your token from: https://quantum.ibm.com/\n"
            "Example .env format: Quantum_IBM=YOUR_TOKEN_HERE"
        )
else:
    # Save the account using the correct channel name
    print(f"\nSaving IBM Quantum account...")
    try:
        QiskitRuntimeService.save_account(
            channel="ibm_quantum", 
            token=Quantum_IBM, 
            overwrite=True
        )
        print(" IBM Quantum account saved successfully!")
    except Exception as e:
        print(f" Error saving with 'ibm_quantum' channel: {e}")
        print("  Trying alternative channel 'ibm_quantum_platform'...")
        try:
            QiskitRuntimeService.save_account(
                channel="ibm_quantum_platform", 
                token=Quantum_IBM, 
                overwrite=True
            )
            print(" IBM Quantum account saved with 'ibm_quantum_platform' channel!")
        except Exception as e2:
            print(f" Failed to save account: {e2}")
            raise
    
    # Load and verify
    print("Loading saved account...")
    service = QiskitRuntimeService()
    print(" IBM Quantum credentials loaded successfully!")
    print(f" Available backends: {len(service.backends())}")
    
    # List available backends
    print("\nAvailable IBM Quantum Backends:")

    for backend in service.backends():
        status = backend.status()
        print(f"  {backend.name:30s} - Qubits: {backend.num_qubits:3d} - Status: {status.status_msg}")


 .env file loaded: True
 Found IBM Quantum token in environment variable: Quantum_IBM
  Token length: 44 characters
Saving IBM Quantum account...
 Error saving with 'ibm_quantum' channel: "Invalid `channel` value. Expected one of ['ibm_cloud', 'ibm_quantum_platform'], got 'ibm_quantum'."
  Trying alternative channel 'ibm_quantum_platform'...
 IBM Quantum account saved with 'ibm_quantum_platform' channel!
Loading saved account...


qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-12-08 17:42:18,205: Default instance not set. Searching all available instances.
qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-12-08 17:42:18,205: Default instance not set. Searching all available instances.


 IBM Quantum credentials loaded successfully!


qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:42:19,207: Unable to create configuration for ibm_fez. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 
qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:42:19,207: Unable to create configuration for ibm_fez. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 
qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:42:19,431: Unable to create configuration for ibm_marrakesh. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 
qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:42:19,431: Unable to create configuration for ibm_marrakesh. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 
qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:42:19,703: Unable to create configuration for ibm_torino. configuration_fro

 Available backends: 0
Available IBM Quantum Backends:


In [21]:
# Paths
DATA_DIR = "/Users/priyanshudey/Code/Qunatum copy/othercode/data"
SAVE_DIR = "./ligand_pocket_comparison_results_IBM"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Results will be saved to: {SAVE_DIR}")

Results will be saved to: ./ligand_pocket_comparison_results_IBM


## 1. Load Data

In [22]:

# Data parameters
MAX_SAMPLES = 0  # Set to None for full dataset

SEED = 42069
SIMULATION_SEED = 42069  # For quantum simulation
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # OPTIMIZATION: Enable benchmark mode for faster training
    torch.backends.cudnn.deterministic = False  # Set to False for speed
    torch.backends.cudnn.benchmark = True  # Set to True for speed
# Initialize Processor
processor = LigandPocketDataProcessor(DATA_DIR, seed=SEED)

# Load Data
processor.load_data(max_samples=MAX_SAMPLES)

interactions = processor.get_dataset()
print(f"Total Interactions: {len(interactions)}")# Reproducibility

Searching for data in: /Users/priyanshudey/Code/Qunatum copy/othercode/data
Found 15239 protein descriptor files


Loading Data: 100%|██████████| 15239/15239 [00:47<00:00, 322.90it/s]


Generating negative samples (target: 122130)...


Generating Negatives: 100%|██████████| 122130/122130 [14:37<00:00, 139.19it/s]

Loaded 13201 pockets, 122130 ligands
Interactions: 122130 positive, 122130 negative
Total Interactions: 244260


In [ ]:
# Model parameters - IBM QUANTUM HARDWARE CONFIGURATION
HIDDEN_DIM = 64
N_QUBITS = 6  # Optimized for IBM quantum computers
N_QLAYERS = 2  # Shallow circuit to minimize noise


IBM_BACKEND = 'ibm_fez'  # *** REAL QUANTUM HARDWARE ***

# Training parameters - OPTIMIZED FOR REAL QUANTUM HARDWARE
PIN_MEMORY = True
EPOCHS = 5  # REDUCED for real hardware - each epoch takes HOURS!
LEARNING_RATE_QUANTUM = 0.001
LEARNING_RATE_CLASSICAL = 0.001
EARLY_STOPPING_PATIENCE = 3  # Reduced patience for faster stopping


print(f"IBM QUANTUM HARDWARE EXPERIMENT CONFIGURATION")

print(f"  IBM Backend: {IBM_BACKEND}")
print(f"  Backend Type: REAL QUANTUM COMPUTER ")
print(f"  Qubits: {N_QUBITS}")
print(f"  Quantum Layers (Depth): {N_QLAYERS}")
print(f"  Hidden Dim: {HIDDEN_DIM}")
print(f"  Epochs: {EPOCHS} (REDUCED for real hardware)")
print(f"  Learning Rate: {LEARNING_RATE_QUANTUM}")
print(f"  Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Data Split: 80% train / 10% val / 10% test")

print(f"\n⚠ REAL QUANTUM HARDWARE SELECTED ")
print(f"  This will run on IBM's {IBM_BACKEND} quantum computer!")
print(f"  Expected time: HOURS to DAYS")
print(f"  Each quantum circuit execution takes seconds to minutes")
print(f"  Total training time estimate: 20-100+ hours")


In [ ]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")

# Auto-detect CPU and GPU
CPU_COUNT = mp.cpu_count()
NUM_WORKERS = max(4, CPU_COUNT - 2)
PREFETCH_FACTOR = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# CRITICAL: Very small batch size for REAL IBM Quantum hardware
# Each sample requires quantum circuit execution on real hardware
# Smaller batches = more manageable queue times and memory usage

BATCH_SIZE = 16  # VERY SMALL for real quantum hardware


print(f"TRAINING CONFIGURATION FOR IBM QUANTUM HARDWARE")

print(f"Device: {DEVICE}")
print(f"IBM Backend: {IBM_BACKEND}")
print(f"CPU Cores: {CPU_COUNT}")
print(f"Data Workers: {NUM_WORKERS}")
print(f"Batch Size: {BATCH_SIZE} (OPTIMIZED for real quantum hardware)")
print(f"Prefetch Factor: {PREFETCH_FACTOR}")

# Calculate training statistics (will be updated after data split)
total_samples = len(interactions)
estimated_train_samples = int(total_samples * 0.8)
estimated_val_samples = int(total_samples * 0.1)
estimated_test_samples = int(total_samples * 0.1)

print(f"\nEstimated Dataset Sizes (80/10/10 split):")
print(f"  Total samples: {total_samples:,}")
print(f"  Train samples: ~{estimated_train_samples:,}")
print(f"  Val samples: ~{estimated_val_samples:,}")
print(f"  Test samples: ~{estimated_test_samples:,}")

# Time estimates for REAL quantum hardware
est_batches_per_epoch = estimated_train_samples // BATCH_SIZE
est_time_per_batch_minutes = 2  # Conservative: 2 minutes per batch
est_epoch_time_hours = (est_batches_per_epoch * est_time_per_batch_minutes) / 60
est_total_time_hours = est_epoch_time_hours * EPOCHS


print(f"TIME ESTIMATES (CONSERVATIVE)")

print(f"  Estimated batches per epoch: ~{est_batches_per_epoch:,}")
print(f"  Estimated time per batch: ~{est_time_per_batch_minutes} minutes")
print(f"  Estimated time per epoch: ~{est_epoch_time_hours:.1f} hours")
print(f"  Estimated TOTAL training: ~{est_total_time_hours:.1f} hours ({est_total_time_hours/24:.1f} days)")
print(f"\n⚠  This is a CONSERVATIVE estimate")
print(f"   Actual time depends on:")
print(f"     - Queue length on {IBM_BACKEND}")
print(f"     - Circuit transpilation time")
print(f"     - Network latency")
print(f"     - Job submission overhead")


In [ ]:
# Create Datasets and Loaders with TRAIN/VAL/TEST SPLIT
# Based on reference implementation: 80% train, 10% val, 10% test

# Step 1: Split into train (80%) and temp (20%)
train_ints, temp_ints = train_test_split(interactions, test_size=0.2, random_state=SEED)

# Step 2: Split temp into val (50%) and test (50%), giving final 10% val, 10% test
val_ints, test_ints = train_test_split(temp_ints, test_size=0.5, random_state=SEED)

# Create datasets
train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)
test_dataset = LigandPocketDataset(processor, test_ints)

# HEAVILY OPTIMIZED DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=optimized_collate_fn,  
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,  
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=optimized_collate_fn,  
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR
)

print(f"✓ DataLoaders created with optimized collate function")
print(f"\nDataset Split (80/10/10):")
print(f"  Train: {len(train_dataset):,} samples ({len(train_loader)} batches)")
print(f"  Val:   {len(val_dataset):,} samples ({len(val_loader)} batches)")
print(f"  Test:  {len(test_dataset):,} samples ({len(test_loader)} batches)")
print(f"\nConfiguration:")
print(f"  Samples per batch: {BATCH_SIZE}")
print(f"  Data loading workers: {NUM_WORKERS}")
print(f"  Prefetch factor: {PREFETCH_FACTOR} batches/worker")
print(f"  Total prefetched batches: {NUM_WORKERS * PREFETCH_FACTOR}")

In [26]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")

Ligand Input Dim: 10
Pocket Input Dim: 19


## 3. Training Functions

In [ ]:
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch with progress tracking."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (x_batch, edge_index_batch, batch_vec, pocket_batch, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        # Non-blocking transfer to GPU
        x_batch = x_batch.to(device, non_blocking=True)
        edge_index_batch = edge_index_batch.to(device, non_blocking=True)
        batch_vec = batch_vec.to(device, non_blocking=True)
        pocket_batch = pocket_batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        # Forward pass
        outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'time': f'{batch_time:.1f}s'})
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}


def evaluate(model, criterion, loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_batch, edge_index_batch, batch_vec, pocket_batch, labels in tqdm(loader, desc='Validation', leave=False):
            # Non-blocking transfer
            x_batch = x_batch.to(device, non_blocking=True)
            edge_index_batch = edge_index_batch.to(device, non_blocking=True)
            batch_vec = batch_vec.to(device, non_blocking=True)
            pocket_batch = pocket_batch.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }


def train_model(model, train_loader, val_loader, learning_rate, model_name, device, resume_from_checkpoint=None):
    """Train with detailed progress tracking and optional resume functionality."""
    
    print(f"Training {model_name.upper()} Model")
    
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Batches per epoch: {len(train_loader)}\n")
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = nn.BCELoss()
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_epoch = 0
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint is not None:
        
        print(f"RESUMING FROM CHECKPOINT")
        
        
        checkpoint_path = os.path.join(SAVE_DIR, f"{model_name}_best.pt")
        history_path = os.path.join(SAVE_DIR, f"{model_name}_history.json")
        
        if os.path.exists(checkpoint_path) and os.path.exists(history_path):
            # Load checkpoint
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            best_val_auc = checkpoint['best_auc']
            
            # Load history
            with open(history_path, 'r') as f:
                history = json.load(f)
            
            start_epoch = len(history['train_loss'])
            
            # Calculate patience counter (epochs since last improvement)
            best_epoch = np.argmax(history['val_auc'])
            patience_counter = start_epoch - 1 - best_epoch
            
            print(f" Loaded checkpoint from epoch {checkpoint['epoch'] + 1}")
            print(f" Best AUC so far: {best_val_auc:.4f} (epoch {best_epoch + 1})")
            print(f" Resuming from epoch {start_epoch + 1}")
            print(f" Current patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            
            # Restore scheduler state by running it on past history
            for auc in history['val_auc']:
                scheduler.step(auc)
            
            print(f" Current learning rate: {optimizer.param_groups[0]['lr']:.6f}")
            print(f"{'='*70}\n")
        else:
            print(f" Checkpoint files not found, starting from scratch")
            print(f"{'='*70}\n")
    
    start_time = datetime.now()
    
    for epoch in range(start_epoch, EPOCHS):
        epoch_start = datetime.now()
        
        print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
        
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        
        # Print results
        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1} Results ({epoch_time:.1f}s):")
        print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
              f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
        print(f"  Best AUC so far: {best_val_auc:.4f}")
        print(f"{'-'*70}")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f" New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        # Save history every epoch
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/60:.2f} minutes")
    print(f"  Best AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc

## 4. Train Quantum Model

In [ ]:
print("="*70)
print("CREATING IBM QUANTUM MODEL")
print("="*70)
print(f"Connecting to IBM Quantum backend: {IBM_BACKEND}")
print(f"This will connect to a REAL quantum computer...")
print("="*70)

quantum_model = LigandPocketQGNN_IBM(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=True,
    ibm_backend=IBM_BACKEND,  # ibm_fez - REAL QUANTUM HARDWARE!
    use_session=True  # Use IBM Runtime sessions for better performance
)


print(f"IBM QUANTUM MODEL ARCHITECTURE")

print(quantum_model)


print(f"\n Starting training on IBM Quantum Hardware: {IBM_BACKEND}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Expected completion: {(datetime.now() + pd.Timedelta(hours=est_total_time_hours)).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n This will take A LONG TIME - consider running overnight!")
print(f"{'='*70}\n")

# Train the model
quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum_ibm_fez",
    DEVICE,
    resume_from_checkpoint=None
)


print(f"IBM QUANTUM TRAINING COMPLETE!")
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Best AUC: {quantum_best_auc:.4f}")


CREATING IBM QUANTUM MODEL
Connecting to IBM Quantum backend: ibm_fez
This will connect to a REAL quantum computer...
IBM QUANTUM BACKEND CONFIGURATION
Initializing IBM Quantum backend: ibm_fez


qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-12-08 17:58:15,516: Default instance not set. Searching all available instances.
qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-12-08 17:58:15,516: Default instance not set. Searching all available instances.


  Using instance: open-instance (open)


qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:58:17,326: Unable to create configuration for ibm_fez. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 
qiskit_runtime_service._create_backend_obj:WARNING:2025-12-08 17:58:17,326: Unable to create configuration for ibm_fez. configuration_from_server_data() got an unexpected keyword argument 'use_fractional_gates' 


 Error connecting to IBM backend: 'No backend matches the criteria. Learn more about available backends here https://quantum.cloud.ibm.com/docs/en/guides/qpu-information#view-your-resources'
  Please ensure your account is saved correctly
  Run in notebook: QiskitRuntimeService.save_account(token='YOUR_IBM_TOKEN', overwrite=True)


QiskitBackendNotFoundError: 'No backend matches the criteria. Learn more about available backends here https://quantum.cloud.ibm.com/docs/en/guides/qpu-information#view-your-resources'

## 5. Train Classical Model

In [ ]:
print("Creating Classical Model...")
from model import LigandPocketQGNN  # Use regular model for classical

classical_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS,
    n_qlayers=N_QLAYERS,
    use_quantum=False,  # Classical version
)

print(f"\nClassical Model Architecture:")
print(classical_model)

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

classical_history, classical_best_auc = train_model(
    classical_model,
    train_loader,
    val_loader,
    LEARNING_RATE_CLASSICAL,
    "classical_ibm",
    DEVICE,
    resume_from_checkpoint=None  # Set to True to resume
)

print(f"\nClassical training finished at: {datetime.now().strftime('%H:%M:%S')}")

## 6. Comparison Results

In [ ]:
print("\n" + "="*70)
print("FINAL COMPARISON: IBM QUANTUM vs CLASSICAL")
print("="*70)
print(f"IBM Quantum Backend: {IBM_BACKEND}")
print(f"Number of Qubits: {N_QUBITS}")
print(f"Circuit Depth: {N_QLAYERS} layers")
print("-"*70)
print(f"IBM Quantum Model - Best AUC: {quantum_best_auc:.4f}")
print(f"Classical Model   - Best AUC: {classical_best_auc:.4f}")
print(f"\nQuantum Advantage: {(quantum_best_auc - classical_best_auc)*100:.2f}% {'improvement' if quantum_best_auc > classical_best_auc else 'deficit'}")
print("="*70)

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'IBM Quantum ({IBM_BACKEND}) vs Classical Comparison', fontsize=16, fontweight='bold')

# Loss
axes[0, 0].plot(quantum_history['train_loss'], label='IBM Quantum Train', color='blue', alpha=0.7)
axes[0, 0].plot(classical_history['train_loss'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(quantum_history['val_loss'], label='IBM Quantum Val', color='blue', alpha=0.7)
axes[1, 0].plot(classical_history['val_loss'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 0].set_title('Validation Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(quantum_history['train_acc'], label='IBM Quantum Train', color='blue', alpha=0.7)
axes[0, 1].plot(classical_history['train_acc'], label='Classical Train', color='orange', alpha=0.7)
axes[0, 1].set_title('Training Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(quantum_history['val_acc'], label='IBM Quantum Val', color='blue', alpha=0.7)
axes[1, 1].plot(classical_history['val_acc'], label='Classical Val', color='orange', alpha=0.7)
axes[1, 1].set_title('Validation Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# AUC
axes[0, 2].plot(quantum_history['val_auc'], label='IBM Quantum', color='blue', marker='o', alpha=0.7)
axes[0, 2].plot(classical_history['val_auc'], label='Classical', color='orange', marker='s', alpha=0.7)
axes[0, 2].axhline(y=quantum_best_auc, color='blue', linestyle='--', alpha=0.5, label=f'IBM Q Best: {quantum_best_auc:.4f}')
axes[0, 2].axhline(y=classical_best_auc, color='orange', linestyle='--', alpha=0.5, label=f'Classical Best: {classical_best_auc:.4f}')
axes[0, 2].set_title('Validation AUC')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('AUC')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# F1 Score
axes[1, 2].plot(quantum_history['val_f1'], label='IBM Quantum', color='blue', alpha=0.7)
axes[1, 2].plot(classical_history['val_f1'], label='Classical', color='orange', alpha=0.7)
axes[1, 2].set_title('Validation F1 Score')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('F1')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f'comparison_plots_{IBM_BACKEND}.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPlots saved to {SAVE_DIR}/comparison_plots_{IBM_BACKEND}.png")

## 8. Final Metrics Summary

In [ ]:
import pandas as pd

# Get best epoch metrics for both models
q_best_idx = np.argmax(quantum_history['val_auc'])
c_best_idx = np.argmax(classical_history['val_auc'])

results_df = pd.DataFrame({
    'Model': ['IBM Quantum', 'Classical'],
    'Backend': [IBM_BACKEND, 'CPU/GPU'],
    'Qubits': [N_QUBITS, 'N/A'],
    'Circuit Depth': [N_QLAYERS, 'N/A'],
    'Best Epoch': [q_best_idx + 1, c_best_idx + 1],
    'Val Loss': [
        quantum_history['val_loss'][q_best_idx],
        classical_history['val_loss'][c_best_idx]
    ],
    'Val Acc': [
        quantum_history['val_acc'][q_best_idx],
        classical_history['val_acc'][c_best_idx]
    ],
    'Val AUC': [quantum_best_auc, classical_best_auc],
    'Val Precision': [
        quantum_history['val_precision'][q_best_idx],
        classical_history['val_precision'][c_best_idx]
    ],
    'Val Recall': [
        quantum_history['val_recall'][q_best_idx],
        classical_history['val_recall'][c_best_idx]
    ],
    'Val F1': [
        quantum_history['val_f1'][q_best_idx],
        classical_history['val_f1'][c_best_idx]
    ]
})

print("\n" + "="*70)
print("SUMMARY OF BEST METRICS - IBM QUANTUM vs CLASSICAL")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Save results
results_df.to_csv(os.path.join(SAVE_DIR, f'comparison_results_{IBM_BACKEND}.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}/comparison_results_{IBM_BACKEND}.csv")

# Save metadata
metadata = {
    'ibm_backend': IBM_BACKEND,
    'n_qubits': N_QUBITS,
    'n_qlayers': N_QLAYERS,
    'batch_size': BATCH_SIZE,
    'hidden_dim': HIDDEN_DIM,
    'learning_rate': LEARNING_RATE_QUANTUM,
    'epochs_trained': len(quantum_history['train_loss']),
    'quantum_best_auc': quantum_best_auc,
    'classical_best_auc': classical_best_auc,
    'quantum_advantage_percent': (quantum_best_auc - classical_best_auc) * 100,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open(os.path.join(SAVE_DIR, f'experiment_metadata_{IBM_BACKEND}.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to {SAVE_DIR}/experiment_metadata_{IBM_BACKEND}.json")